# ForgeAI Colab T4 Ollama + ngrok Server

This notebook runs an Ollama-compatible local LLM on a Colab **T4 GPU** and exposes it through an ngrok HTTPS tunnel. ForgeAI can then use the tunnel from your laptop with `--provider ollama --ollama-url <ngrok-url>`.

## Security note

Ollama has no built-in auth. An ngrok URL is a public URL. Keep it private, stop the tunnel when done, and do not use it for sensitive prompts.


## 1. Confirm the GPU runtime

In Colab, choose **Runtime → Change runtime type → T4 GPU** before running this cell.


In [ ]:
!nvidia-smi


## 2. Install Ollama and ngrok helpers

This installs the Ollama server and `pyngrok`. You need a free ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken. Store it in Colab Secrets as `NGROK_AUTHTOKEN`, or paste it when prompted.


In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install pyngrok requests


## 3. Start Ollama on all interfaces

Colab needs Ollama listening on `0.0.0.0:11434` so ngrok can forward to it.


In [ ]:
import os
import subprocess
import time

ollama_env = os.environ.copy()
ollama_env["OLLAMA_HOST"] = "0.0.0.0:11434"
ollama_env["OLLAMA_ORIGINS"] = "*"

server = subprocess.Popen(["ollama", "serve"], env=ollama_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(5)
print("Ollama server started with pid", server.pid)
!curl -s http://127.0.0.1:11434/api/tags


## 4. Pull a T4-friendly model

`qwen2.5:7b-instruct` is a practical starting point for JSON-heavy ForgeAI prompts on a T4. Larger models may be slower or run out of memory.


In [ ]:
MODEL = "qwen2.5:7b-instruct"
!ollama pull {MODEL}
!ollama list


## 5. Smoke test the model locally inside Colab


In [ ]:
import json
import requests

payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Return only JSON: {\"ok\": true}"}],
    "stream": False,
    "format": "json",
}
response = requests.post("http://127.0.0.1:11434/api/chat", json=payload, timeout=120)
print(response.status_code)
print(response.text[:1000])


## 6. Open an ngrok tunnel

Copy the printed public URL. Use it as ForgeAI's `--ollama-url`. Keep this Colab runtime alive while generating.


In [ ]:
import getpass
from pyngrok import ngrok, conf

try:
    from google.colab import userdata
    ngrok_token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    ngrok_token = None

if not ngrok_token:
    ngrok_token = getpass.getpass("Paste NGROK_AUTHTOKEN: ")

conf.get_default().auth_token = ngrok_token
try:
    ngrok.kill()
except Exception:
    pass

tunnel = ngrok.connect(11434, "http")
OLLAMA_PUBLIC_URL = tunnel.public_url
print("OLLAMA_PUBLIC_URL=" + OLLAMA_PUBLIC_URL)
print("Model=" + MODEL)


## 7. Test the public tunnel from Colab


In [ ]:
public_tags = requests.get(OLLAMA_PUBLIC_URL + "/api/tags", timeout=30)
print(public_tags.status_code)
print(public_tags.text[:1000])


## 8. Use this server from ForgeAI

On your local machine, run:

```bash
export FORGEAI_OLLAMA_BASE_URL="https://YOUR-NGROK-URL.ngrok-free.app"
uefn-ai doctor --provider ollama --model qwen2.5:7b-instruct --ollama-url "$FORGEAI_OLLAMA_BASE_URL"

uefn-ai create "A compact lumber tycoon for 4 players with one upgrade lane and worker automation." \
  --provider ollama \
  --model qwen2.5:7b-instruct \
  --ollama-url "$FORGEAI_OLLAMA_BASE_URL" \
  --genre tycoon \
  --template tycoon/lumber-mill \
  --seed 101 \
  --out ./output/colab-t4-lumber \
  --budget 0.01 \
  --zip
```

The budget can be very low because local/Ollama calls are priced at `$0` in ForgeAI's ledger. It is still useful as a guardrail if fallback providers are enabled.


## 9. Stop the tunnel when done


In [ ]:
# Run this when finished.
# ngrok.kill()
# server.terminate()
